# BEM layers from CHARM parcellation

## 🧪 - Experiment setup
- `seg`: CHARM head segmentation

In [ ]:
# imports
import ants
import mne
import mindmap as mm

# readers
dm = mm.io.TutoManager("../../config.yml")
sub = "simf10"

# data
brain = mm.anatomy.mri.load_fs_in_native_space(dm.roots["fs"]/f"sub-{sub}", volume_type="brain")
charm = dm.load(sub, "seg")
ras2tkras = dm.load(sub, "ras2tkras")    # to build meshes in FreeSurfer space (expected by MNE)
vol_info = dm.load(sub, "vol-info")      # metadata for writing surfaces
bem_labels = dm.get_bem_labels()


## 🧬 - Data Processing
Build Boundary Element Model

In [ ]:
# brain surface
brain_mask = mm.anatomy.mri.nibabel_to_ants(brain)
brain_mask = ants.threshold_image(brain_mask, low_thresh=0.5, high_thresh=None, inval=1, outval=0)
brain_mask = brain_mask.morphology("close", radius=10)

brain_surf, brain_interop = mm.anatomy.forward.fit_surface_to_mri_mask(brain_mask, n_smooth=20)
brain_surf = mm.geometry.polydata_transform(brain_surf, ras2tkras)

# build BEM layers from charm segmentation
layers = mm.anatomy.forward.build_bem_layers(charm, bem_labels, n_smooth=20)
layers = mm.anatomy.forward.bem_transform(layers, ras2tkras)

# save layers in freesurfer/{subject}/bem
folder = dm.roots["fs"]/f"sub-{sub}"/"bem"
folder.mkdir(exist_ok=True)
mm.anatomy.forward.save_bem_as_freesurfer(folder, layers, vol_info, overwrite=True)

# assemble
model = mne.make_bem_model(
    subject      = f"sub-{sub}",
    subjects_dir = dm.roots["fs"],
    conductivity = [0.3, 0.006, 0.3],    # conductivities (brain, skull, skin)
    ico          = None,                 # mesh resolution (simplifies computations), tris don't match though
)

bem  = mne.make_bem_solution(model)

Overwriting existing file.
Overwriting existing file.
Overwriting existing file.


## 📊 - Visualization

Visualize in freeview

In [3]:
mm.viz.plot_bem(dm.roots["fs"], subject="simf10")

Visualize 3D

In [9]:
import pyvista as pv
p = pv.Plotter()
p.add_mesh(brain_surf, style="wireframe", color="pink", line_width=1)
p.add_mesh(layers["skin"], style="wireframe", color="green", line_width=1)
p.add_mesh(layers["outer skull"], style="wireframe", color="black", line_width=1)
p.add_mesh(layers["inner skull"], style="wireframe", color="red", line_width=1)
p.show()

Widget(value='<iframe src="http://localhost:46689/index.html?ui=P_0x7ae50e22e390_1&reconnect=auto" class="pyvi…